# Phase 3: Macroeconomic Stress Testing Engine
### Point-in-Time Multi-Scenario Macro Shocks & PD Sensitivity Analysis

**Objective:**  
Subject the isolated recent loan portfolio (517,807 loans from 2016–2018) to multi-tier macroeconomic shocks formulated under regulatory stress testing frameworks (CCAR / DFAST / CECL):
1. **Baseline Scenario:** Prevailing historical macroeconomic numbers $\rightarrow \text{PD}_{\text{base}}$
2. **Adverse Scenario (Mild Recession):** $\text{UNRATE} + 1.5\%$, $\text{FEDFUNDS} + 0.5\% \rightarrow \text{PD}_{\text{adverse}}$
3. **Severe Scenario (Deep Recession):** $\text{UNRATE} + 3.5\%$, $\text{FEDFUNDS} + 1.5\% \rightarrow \text{PD}_{\text{severe}}$

---

### Macro Shock Specification Table
| Scenario | Unemployment Rate Shock ($\Delta \text{UNRATE}$) | Fed Funds Rate Shock ($\Delta \text{FEDFUNDS}$) | Economic Narrative |
| :--- | :---: | :---: | :--- |
| **Baseline** | $+0.0\%$ | $+0.0\%$ | Prevailing baseline economy |
| **Adverse** | $+1.5\%$ | $+0.5\%$ | Moderate economic downturn & rate hike |
| **Severe** | $+3.5\%$ | $+1.5\%$ | Severe stagflationary recessionary shock |

In [ ]:
# Cell 1: Environment Setup & Load Champion Model Pipeline
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

model_path = "models/champion_pd_model.joblib"
print(f"Loading champion model from {model_path}...")
pipeline = joblib.load(model_path)
print("Model pipeline loaded successfully.")

In [ ]:
# Cell 2: Load Isolated Test Portfolio (2016-2018 Loans)
portfolio_path = "data/test_portfolio_with_pd.parquet"
print(f"Loading isolated test portfolio from {portfolio_path}...")
df_portfolio = pd.read_parquet(portfolio_path)
print(f"Test Portfolio Size: {len(df_portfolio):,} loans | Total Loan Balance: ${df_portfolio['loan_amnt'].sum():,.2f}")
display(df_portfolio[['loan_amnt', 'fico_range_low', 'dti', 'purpose', 'UNRATE', 'FEDFUNDS', 'PD_base']].head())

In [ ]:
# Cell 3: Execute Scenario 1 (Baseline PD)
features = ['fico_range_low', 'dti', 'purpose', 'UNRATE', 'FEDFUNDS']
X_base = df_portfolio[features].copy()

t0 = time.time()
df_portfolio['PD_base'] = pipeline.predict_proba(X_base)[:, 1].astype(np.float32)
print(f"Baseline PD calculated in {time.time() - t0:.2f}s")
print(f"Mean Baseline PD: {df_portfolio['PD_base'].mean()*100:.2f}%")

In [ ]:
# Cell 4: Execute Scenario 2 (Adverse Shock: +1.5% UNRATE, +0.5% FEDFUNDS)
X_adverse = df_portfolio[features].copy()
X_adverse['UNRATE'] = X_adverse['UNRATE'] + 1.5
X_adverse['FEDFUNDS'] = X_adverse['FEDFUNDS'] + 0.5

t0 = time.time()
df_portfolio['PD_adverse'] = pipeline.predict_proba(X_adverse)[:, 1].astype(np.float32)
print(f"Adverse PD calculated in {time.time() - t0:.2f}s")
print(f"Mean Adverse PD: {df_portfolio['PD_adverse'].mean()*100:.2f}%")
print(f"Adverse PD Relative Increase: {((df_portfolio['PD_adverse'].mean() - df_portfolio['PD_base'].mean()) / df_portfolio['PD_base'].mean())*100:+.2f}%")

In [ ]:
# Cell 5: Execute Scenario 3 (Severe Shock: +3.5% UNRATE, +1.5% FEDFUNDS)
X_severe = df_portfolio[features].copy()
X_severe['UNRATE'] = X_severe['UNRATE'] + 3.5
X_severe['FEDFUNDS'] = X_severe['FEDFUNDS'] + 1.5

t0 = time.time()
df_portfolio['PD_severe'] = pipeline.predict_proba(X_severe)[:, 1].astype(np.float32)
print(f"Severe PD calculated in {time.time() - t0:.2f}s")
print(f"Mean Severe PD: {df_portfolio['PD_severe'].mean()*100:.2f}%")
print(f"Severe PD Relative Increase: {((df_portfolio['PD_severe'].mean() - df_portfolio['PD_base'].mean()) / df_portfolio['PD_base'].mean())*100:+.2f}%")

In [ ]:
# Cell 6: Multi-Scenario Distribution & Risk Delta Analysis
df_portfolio['PD_adverse_delta'] = df_portfolio['PD_adverse'] - df_portfolio['PD_base']
df_portfolio['PD_severe_delta'] = df_portfolio['PD_severe'] - df_portfolio['PD_base']

scenario_summary = pd.DataFrame({
    'Metric': ['Mean PD (%)', 'Median PD (%)', '25th Percentile (%)', '75th Percentile (%)', '95th Percentile (%)'],
    'Baseline': [
        df_portfolio['PD_base'].mean()*100,
        df_portfolio['PD_base'].median()*100,
        df_portfolio['PD_base'].quantile(0.25)*100,
        df_portfolio['PD_base'].quantile(0.75)*100,
        df_portfolio['PD_base'].quantile(0.95)*100
    ],
    'Adverse (+1.5% U / +0.5% R)': [
        df_portfolio['PD_adverse'].mean()*100,
        df_portfolio['PD_adverse'].median()*100,
        df_portfolio['PD_adverse'].quantile(0.25)*100,
        df_portfolio['PD_adverse'].quantile(0.75)*100,
        df_portfolio['PD_adverse'].quantile(0.95)*100
    ],
    'Severe (+3.5% U / +1.5% R)': [
        df_portfolio['PD_severe'].mean()*100,
        df_portfolio['PD_severe'].median()*100,
        df_portfolio['PD_severe'].quantile(0.25)*100,
        df_portfolio['PD_severe'].quantile(0.75)*100,
        df_portfolio['PD_severe'].quantile(0.95)*100
    ]
})

display(scenario_summary.round(2))

# Visualizing Scenario Shift Distributions
plt.figure(figsize=(10, 5))
plt.hist(df_portfolio['PD_base']*100, bins=50, alpha=0.5, label='Baseline Scenario', color='green', density=True)
plt.hist(df_portfolio['PD_adverse']*100, bins=50, alpha=0.5, label='Adverse Scenario (+1.5% UNRATE)', color='orange', density=True)
plt.hist(df_portfolio['PD_severe']*100, bins=50, alpha=0.5, label='Severe Scenario (+3.5% UNRATE)', color='red', density=True)
plt.title('Probability of Default (PD) Distribution Across Stress Scenarios')
plt.xlabel('Probability of Default (%)')
plt.ylabel('Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 7: Export Stressed Portfolio for Phase 4 Financial Math & SQL Analysis
output_csv_path = "data/stressed_portfolio_phase3.csv"
output_parquet_path = "data/stressed_portfolio_phase3.parquet"

# Ensure clean column selection for SQL import
sql_cols = [
    'Year_Month', 
    'loan_amnt', 
    'fico_range_low', 
    'dti', 
    'purpose', 
    'UNRATE', 
    'FEDFUNDS', 
    'PD_base', 
    'PD_adverse', 
    'PD_severe'
]

df_export = df_portfolio[sql_cols].copy()

# Export to Parquet and CSV
df_export.to_parquet(output_parquet_path, index=False)
df_export.to_csv(output_csv_path, index=False)

print(f"Stressed Portfolio Parquet saved to: {output_parquet_path} ({os.path.getsize(output_parquet_path)/(1024*1024):.2f} MB)")
print(f"Stressed Portfolio CSV saved to: {output_csv_path} ({os.path.getsize(output_csv_path)/(1024*1024):.2f} MB)")
print("\nPhase 3 Complete: Ready for Phase 4 Financial Math (SQL Expected Credit Loss)!")